# NBL-survival
For response to reviewers.
Is there any difference in survival between extra- and intrachromosomally amplified neuroblastomas?

## Dependencies
Dependencies:  
r-survival.yml  
Run `preprocess-rodriguezfos-data.ipynb`

## Results
ecDNA has (barely) significantly worse outcomes than intrachromosomal amplification, both in the
Kaplan-Meier model (adjusted p = 0.042) and in the Cox model controlling for sex, age and amplification ONLY (p = 0.048). Including MYCN amplification as a covariate, using either the Rodriguez annotations or the AmpliconClassifier, renders the effect of ecDNA nonsignificant. Figures saved to `./out`.

## TODO
- Association test comparing incidence of ecDNA vs chromosomal amp for MYCN vs other loci?
- Follow up with Elias and Anton about discrepant ecDNA classifications.

In [ ]:
Sys.setenv(LANGUAGE = "en") # set language to "ja" if you prefer

library(tidyverse)
library(readxl)
library(dplyr)
library(stringr)
library(naniar) #for replace with Nas function
library(survival)
library(survminer)
library(RColorBrewer)
library(janitor)
library(gt)
library(gtsummary)
library(ggsurvfit)
library(extrafont)
library(svglite)

extrafont::font_import(pattern="Arial",prompt=FALSE)
extrafont::loadfonts()

# imports from external file
imports <- new.env()
source("survival-data-imports.R", local = imports)
plotting <- new.env()
source("survival-plots.R",local=plotting)

sessionInfo()

In [ ]:
dir.create('nbl', showWarnings = FALSE)

In [ ]:
nbl_survival_data = 'nbl/processed_nbl_survival_data.tsv'
data <- imports$load_nbl_data(nbl_survival_data)

In [ ]:
data %>%head()

In [ ]:
# KM
formula = Surv(OS_months_5y, OS_status_5y) ~ amplicon_class
km = survfit2(formula=formula, data = data )
plt <- plotting$km_plot(km, "km_nbl_5year")
plotting$km_plot(km)
logrank <- pairwise_survdiff(formula,data,p.adjust.method="BH",rho=0)
logrank

In [ ]:
# Cox model
m4 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amplified + sex + age_at_diagnosis, data = data)
plotting$cox_plot(m4,data,"cox_forest_nbl",width=6,height=3)

In [ ]:
# Cox model including MYCN amp, using rodriguez annotations
m5 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amplified + MYCN_amp + sex + age_at_diagnosis, data = data)
plotting$cox_plot(m5,data,NULL,width=6,height=6)

In [ ]:
# KM of MYCN_amp_AC x ecDNA
data_subset <- data %>% 
    filter(amplified == TRUE) %>%
    mutate(group = case_when(
        ecDNA_status == "ecDNA-" & MYCN_amp_AC == FALSE ~ "chr+ MYCN-",
        ecDNA_status == "ecDNA-" & MYCN_amp_AC == TRUE ~ "chr+ MYCN+",
        ecDNA_status == "ecDNA+" & MYCN_amp_AC == FALSE ~ "ecDNA+ MYCN-",
        ecDNA_status == "ecDNA+" & MYCN_amp_AC == TRUE ~ "ecDNA+ MYCN+"
    ))
formula = Surv(OS_months_5y, OS_status_5y) ~ group
km = survfit2(formula=formula, data = data_subset )
plt <- km_plot(km, "km_nbl_ec_x_mycn")
plt
logrank <- pairwise_survdiff(formula,data_subset,p.adjust.method="BH",rho=0)
logrank
names(km$strata)

In [ ]:
# Cox model including MYCN amp, using AmpliconClassifier annotations
m6 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amplified + MYCN_amp_AC + sex + age_at_diagnosis, data = data)
plotting$cox_plot(m6,data,'cox_forest_nbl_mycn',width=6,height=4)